In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# ==========================================
# 1. READ IMAGE
# ==========================================
img = cv2.imread("/content/bird.jpg")

if img is None:
    print("Image not found")
else:

    # ==========================================
    # 2. RESIZE IMAGE
    # ==========================================
    img = cv2.resize(img, (600, 500))

    # Original Copy
    original = img.copy()

    print("Original Image")
    cv2_imshow(original)

    # ==========================================
    # 3. CONVERT TO GRAYSCALE
    # ==========================================
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    print("Gray Image")
    cv2_imshow(gray)

    # ==========================================
    # 4. REMOVE NOISE
    # ==========================================
    blur = cv2.GaussianBlur(gray, (7,7), 0)

    print("Blurred Image")
    cv2_imshow(blur)

    # ==========================================
    # 5. EDGE DETECTION
    # ==========================================
    edges = cv2.Canny(blur, 50, 150)

    print("Edge Detection")
    cv2_imshow(edges)

    # ==========================================
    # 6. MORPHOLOGICAL OPERATIONS
    # ==========================================
    kernel = np.ones((3,3), np.uint8)

    dilated = cv2.dilate(edges, kernel, iterations=1)
    eroded = cv2.erode(dilated, kernel, iterations=1)

    print("Morphological Processing")
    cv2_imshow(eroded)

    # ==========================================
    # 7. FIND CONTOURS
    # ==========================================
    contours, hierarchy = cv2.findContours(
        eroded,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contour_img = original.copy()

    # ==========================================
    # 8. DRAW CONTOURS + AREA FILTERING
    # ==========================================
    for cnt in contours:

        area = cv2.contourArea(cnt)

        # Ignore small noisy contours
        if area > 500:

            # Draw contour
            cv2.drawContours(contour_img, [cnt], -1, (0,255,0), 2)

            # Bounding Rectangle
            x, y, w, h = cv2.boundingRect(cnt)

            cv2.rectangle(
                contour_img,
                (x,y),
                (x+w, y+h),
                (255,0,0),
                2
            )

            # Display Area
            cv2.putText(
                contour_img,
                f"Area: {int(area)}",
                (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,0,255),
                2
            )

            # Crop ROI
            roi = original[y:y+h, x:x+w]

            print("Detected Object")
            cv2_imshow(roi)

    # ==========================================
    # 9. FINAL OUTPUT
    # ==========================================
    print("Final Detection")
    cv2_imshow(contour_img)